In [1]:
import pandas as pd
import numpy as np
import re

import pgeocode
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [2]:
path_folder = "Datasets/"
path_adjuster = "CatRosterReportDayOf_for_distribution_cleaned.xlsx"

In [3]:
ds_adjuster_main = pd.read_excel(
    path_folder+path_adjuster,
    sheet_name=0
)

In [4]:
ref_url = "https://simplemaps.com/static/data/us-cities/uscitiesv1.4.csv"
ref = pd.read_csv(ref_url)

ref_best = (
    ref[["city", "state_id", "population"]]
      .rename(columns={"city": "City", "state_id": "State"})
      .assign(City=lambda d: d["City"].astype(str).str.strip())
      .sort_values("population", ascending=False)
      .drop_duplicates(subset=["City"])
      .drop(columns=["population"])
)

In [5]:
VIRTUAL_RE = re.compile(r"^\s*(?P<state>.+?)\s*-\s*Virtual\s*$", flags=re.IGNORECASE)

def parse_location(location):
    if pd.isna(location):
        return pd.Series([None, None])

    s = str(location).strip()

    m = VIRTUAL_RE.match(s)
    if m:
        return pd.Series(["Virtual", m.group("state").strip()])

    city_part = re.split(r"\s*-\s*", s, maxsplit=1)[0].strip()

    if "," in city_part:
        city_part = city_part.split(",", 1)[0].strip()

    return pd.Series([city_part, None])

ds_adjuster_main[["City_raw", "State_virtual_raw"]] = ds_adjuster_main["Location"].apply(parse_location)

In [6]:
CITY_FIX = {
    "Overland Pk": "Overland Park",
    "RanchoCordova": "Rancho Cordova",
    "New York City": "New York",
    "St. Paul": "Saint Paul",
    "Hamden Law": "Hamden",
    "Jacksonville Law": "Jacksonville",
    "Oklahoma City Law": "Oklahoma City",
    "Providence Law": "Providence",
}

def clean_city(city_raw):
    if pd.isna(city_raw):
        return None
    c = str(city_raw).strip()

    # Strip trailing " Law" for cases you didn't list
    c = re.sub(r"\s+Law\s*$", "", c, flags=re.IGNORECASE)

    # Apply explicit fixes
    c = CITY_FIX.get(c, c)

    return c

ds_adjuster_main["City"] = ds_adjuster_main["City_raw"].apply(clean_city)

ds_adjuster_main.loc[
    ds_adjuster_main["Location"].eq("Melville NY Corp Center Dr"),
    "City"
] = "Melville"


In [7]:
STATE_ABBR = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
    "MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
    "SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
}

STATE_NAME_TO_ABBR = {
    "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA","Colorado":"CO","Connecticut":"CT",
    "Delaware":"DE","Florida":"FL","Georgia":"GA","Hawaii":"HI","Idaho":"ID","Illinois":"IL","Indiana":"IN","Iowa":"IA",
    "Kansas":"KS","Kentucky":"KY","Louisiana":"LA","Maine":"ME","Maryland":"MD","Massachusetts":"MA","Michigan":"MI",
    "Minnesota":"MN","Mississippi":"MS","Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV","New Hampshire":"NH",
    "New Jersey":"NJ","New Mexico":"NM","New York":"NY","North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK",
    "Oregon":"OR","Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC","South Dakota":"SD","Tennessee":"TN","Texas":"TX",
    "Utah":"UT","Vermont":"VT","Virginia":"VA","Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY",
    "District of Columbia":"DC"
}

def normalize_virtual_state(x):
    if pd.isna(x):
        return None
    t = str(x).strip()

    # already an abbreviation?
    if t.upper() in STATE_ABBR:
        return t.upper()
        
    key = t.title()
    return STATE_NAME_TO_ABBR.get(key)

ds_adjuster_main["State_from_virtual"] = ds_adjuster_main["State_virtual_raw"].apply(normalize_virtual_state)

In [8]:
ds_adjuster_main = ds_adjuster_main.merge(
    ref_best,
    on="City",
    how="left",
    suffixes=("", "_ref")
)

ds_adjuster_main["State"] = ds_adjuster_main["State_from_virtual"].combine_first(ds_adjuster_main["State"])

In [9]:
FINAL_STATE_FIX = {
    "Hunt Valley": "MD",
    "West Bridgewater": "MA",
}

mask_fix = ds_adjuster_main["State"].isna() & ds_adjuster_main["City"].isin(FINAL_STATE_FIX)
ds_adjuster_main.loc[mask_fix, "State"] = ds_adjuster_main.loc[mask_fix, "City"].map(FINAL_STATE_FIX)

In [10]:
missing = (
    ds_adjuster_main.loc[ds_adjuster_main["State"].isna(), "Location"]
      .dropna()
      .value_counts()
)

missing.head(30)

Series([], Name: count, dtype: int64)

In [11]:
state_city_counts = (
    ds_adjuster_main
    .groupby(["State", "City"], dropna=False)
    .size()
    .reset_index(name="Count")
    .sort_values(["State", "Count"], ascending=[True, False])
)

state_city_counts[state_city_counts["City"]=="Virtual"]

,State,City,Count
15,DE,Virtual,1
31,MA,Virtual,2
44,ND,Virtual,1
68,SC,Virtual,19
81,VT,Virtual,1


In [12]:
ds_adjuster_main[["TIES Id", "City", "State"]]

,TIES Id,City,State
0,47334245,Saint Paul,MN
1,43992727,West Des Moines,IA
2,45899542,Saint Paul,MN
3,41040719,Overland Park,KS
4,92150926,Maryland Heights,MO
...,...,...,...
769,71276938,West Bridgewater,MA
770,87025921,Knoxville,TN
771,71010442,Melville,LA
772,62280861,Melville,LA


In [13]:
def normalize_city_state(city, state) -> str | None:
    if pd.isna(city) or pd.isna(state):
        return None
    c = str(city).strip()
    s = str(state).strip()
    if not c or not s:
        return None
    return f"{c}, {s}, USA"

def add_lat_lon_from_city_state(
    df: pd.DataFrame,
    city_col: str = "City",
    state_col: str = "State",
    lat_col: str = "Lat",
    lon_col: str = "Lon",
    keep_query_col: bool = True,
    user_agent: str = "ds-geocode-city-state",
    min_delay_seconds: float = 1.0,  # be polite to Nominatim
) -> pd.DataFrame:

    out = df.copy()

    query_col = f"{city_col}_{state_col}__query"
    out[query_col] = out.apply(lambda r: normalize_city_state(r[city_col], r[state_col]), axis=1)

    unique_queries = (
        out[query_col]
        .dropna()
        .unique()
        .tolist()
    )

    geolocator = Nominatim(user_agent=user_agent, timeout=10)
    geocode = RateLimiter(
        geolocator.geocode,
        min_delay_seconds=min_delay_seconds,
        swallow_exceptions=True
    )

    cache = {}
    lat_map = {}
    lon_map = {}

    for q in unique_queries:
        if q in cache:
            loc = cache[q]
        else:
            loc = geocode(q)
            cache[q] = loc

        if loc is None:
            lat_map[q] = np.nan
            lon_map[q] = np.nan
        else:
            lat_map[q] = float(loc.latitude)
            lon_map[q] = float(loc.longitude)

    out[lat_col] = out[query_col].map(lat_map).astype(float)
    out[lon_col] = out[query_col].map(lon_map).astype(float)

    if not keep_query_col:
        out.drop(columns=[query_col], inplace=True)

    return out

ds_adjuster_main = ds_adjuster_main[
    ["TIES Id","CL Skill Level","PL Skill Level","Will Travel","City","State"]
].copy()

ds_adjuster_main = add_lat_lon_from_city_state(
    ds_adjuster_main,
    city_col="City",
    state_col="State",
    lat_col="Lat.Home",
    lon_col="Lon.Home",
    keep_query_col=True,
    min_delay_seconds=1.0
)

print("Missing Lat:", ds_adjuster_main["Lat.Home"].isna().mean())
print("Missing Lon:", ds_adjuster_main["Lon.Home"].isna().mean())

failed = ds_adjuster_main[ds_adjuster_main["Lat.Home"].isna() | ds_adjuster_main["Lon.Home"].isna()]
failed[["City", "State"]].drop_duplicates().head(20)

Missing Lat: 0.0
Missing Lon: 0.0


,City,State


In [14]:
ds_adjuster_main.drop(columns=["City_State__query"], inplace=True)

In [15]:
ds_adjuster_main

,TIES Id,CL Skill Level,PL Skill Level,Will Travel,City,State,Lat.Home,Lon.Home
0,47334245,0,0,,Saint Paul,MN,44.949749,-93.093103
1,43992727,0,0,N,West Des Moines,IA,41.564448,-93.759406
2,45899542,0,1,Y,Saint Paul,MN,44.949749,-93.093103
3,41040719,0,1,Y,Overland Park,KS,38.974250,-94.685170
4,92150926,0,1,Y,Maryland Heights,MO,38.715051,-90.435999
...,...,...,...,...,...,...,...,...
769,71276938,1,2,Y,West Bridgewater,MA,42.019092,-71.007969
770,87025921,3,2,Y,Knoxville,TN,35.960395,-83.921026
771,71010442,0,1,Y,Melville,LA,30.692965,-91.744004
772,62280861,1,4,N,Melville,LA,30.692965,-91.744004


In [16]:
ds_adjuster_main["Will Travel"] = (
    ds_adjuster_main["Will Travel"]
    .astype(str)
    .str.strip()
    .map({"Y": 1, "N": 0, "": 0})
    .fillna(0)
    .astype(int)
)

In [17]:
ds_adjuster_main = ds_adjuster_main[
    (ds_adjuster_main["PL Skill Level"] > 0) | 
    (ds_adjuster_main["CL Skill Level"] > 0)
]

In [18]:
ds_adjuster_main

,TIES Id,CL Skill Level,PL Skill Level,Will Travel,City,State,Lat.Home,Lon.Home
2,45899542,0,1,1,Saint Paul,MN,44.949749,-93.093103
3,41040719,0,1,1,Overland Park,KS,38.974250,-94.685170
4,92150926,0,1,1,Maryland Heights,MO,38.715051,-90.435999
6,44312033,0,3,1,Austin,TX,30.271129,-97.743700
8,35541067,0,1,1,Naperville,IL,41.772870,-88.147928
...,...,...,...,...,...,...,...,...
768,43369495,1,1,1,Albany,NY,42.651167,-73.754968
769,71276938,1,2,1,West Bridgewater,MA,42.019092,-71.007969
770,87025921,3,2,1,Knoxville,TN,35.960395,-83.921026
771,71010442,0,1,1,Melville,LA,30.692965,-91.744004


In [19]:
ds_adjuster_main.to_excel(
    "Datasets/ds_adjuster_optClean.xlsx",
    index=False
)